## Обучение RuBERT для классификации тональности с учётом grad accumulation и warmup
Скрипт загружает размеченные данные, инициализирует модель `DeepPavlov/rubert-base-cased` для трёхклассовой классификации, формирует датасеты и DataLoader’ы и обучает модель с использованием градиентного накопления, линейного scheduler’а с warmup и мониторингом метрик (loss, accuracy, mac


In [ ]:
import os
import time
import random
import math
from typing import List, Dict

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

TRAIN_CLEAN_PATH = "../data/processed/train_super.csv"
VAL_CLEAN_PATH   = "../data/processed/val_super.csv"

MODEL_ID   = "DeepPavlov/rubert-base-cased"
OUTPUT_DIR = "../models/rurorberta_base2_manual"

NUM_LABELS         = 3
MAX_LEN            = 192     
EPOCHS             = 2       
BATCH_SIZE         = 8       
GRAD_ACCUM_STEPS   = 4       
LEARNING_RATE      = 2e-5    
WARMUP_RATIO       = 0.06    
RANDOM_STATE       = 42

MAX_TRAIN_SAMPLES  = None  


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class TextDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_len: int):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }


def train_one_epoch(
    model,
    dataloader,
    optimizer,
    scheduler,
    device,
    grad_accum_steps: int = 1,
    epoch_idx: int = 1,
):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    optimizer.zero_grad()

    num_batches = len(dataloader)
    start_time = time.time()

    for step, batch in enumerate(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )

        loss = outputs.loss / grad_accum_steps
        logits = outputs.logits

        loss.backward()
        total_loss += loss.item() * grad_accum_steps  # считаем "настоящий" лосс

        if (step + 1) % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            if device.type == "mps":
                torch.mps.empty_cache()

        preds = torch.argmax(logits, dim=-1).detach().cpu().numpy()
        labels = batch["labels"].detach().cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels)

        # прогресс каждые 200 батчей
        if (step + 1) % 200 == 0 or (step + 1) == num_batches:
            elapsed = (time.time() - start_time) / 60
            avg_loss_so_far = total_loss / (step + 1)
            progress = 100.0 * (step + 1) / num_batches
            lr = scheduler.get_last_lr()[0]
            print(
                f"[Train] Epoch {epoch_idx} | "
                f"batch {step+1}/{num_batches} ({progress:.1f}%) | "
                f"avg_loss: {avg_loss_so_far:.4f} | lr: {lr:.6f} | "
                f"elapsed: {elapsed:.1f} мин"
            )

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1


def eval_one_epoch(model, dataloader, device, epoch_idx: int = 1):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    num_batches = len(dataloader)
    start_time = time.time()

    with torch.no_grad():
        for step, batch in enumerate(dataloader):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=-1).detach().cpu().numpy()
            labels = batch["labels"].detach().cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels)

            if (step + 1) % 200 == 0 or (step + 1) == num_batches:
                elapsed = (time.time() - start_time) / 60
                avg_loss_so_far = total_loss / (step + 1)
                progress = 100.0 * (step + 1) / num_batches
                print(
                    f"[Val] Epoch {epoch_idx} | "
                    f"batch {step+1}/{num_batches} ({progress:.1f}%) | "
                    f"avg_loss: {avg_loss_so_far:.4f} | "
                    f"elapsed: {elapsed:.1f} мин"
                )

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1


def main():
    set_seed(RANDOM_STATE)

    assert os.path.exists(TRAIN_CLEAN_PATH), f"Нет файла {TRAIN_CLEAN_PATH}"
    assert os.path.exists(VAL_CLEAN_PATH),   f"Нет файла {VAL_CLEAN_PATH}"

    print("Читаем данные...")
    train_df = pd.read_csv(TRAIN_CLEAN_PATH)
    val_df   = pd.read_csv(VAL_CLEAN_PATH)

    for col in ["text", "label"]:
        if col not in train_df.columns:
            raise ValueError(f"В train_super нет колонки '{col}'")
        if col not in val_df.columns:
            raise ValueError(f"В val_super нет колонки '{col}'")

    if MAX_TRAIN_SAMPLES is not None and len(train_df) > MAX_TRAIN_SAMPLES:
        print(f"train_super имеет {len(train_df)} строк, "
              f"режем до {MAX_TRAIN_SAMPLES} для обучения...")
        train_df = train_df.sample(
            n=MAX_TRAIN_SAMPLES,
            random_state=RANDOM_STATE
        ).reset_index(drop=True)

    print("Размер train:", train_df.shape)
    print("Размер val:  ", val_df.shape)
    print("\nРаспределение классов (train):")
    print(train_df["label"].value_counts().sort_index())
    print("\nРаспределение классов (val):")
    print(val_df["label"].value_counts().sort_index())

    # Выбор устройства
    if torch.backends.mps.is_available():
        device = torch.device("mps")     # Mac M1/M2/M3/M4
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print("\nИспользуем устройство:", device)

    print("\nЗагружаем токенизатор и модель:", MODEL_ID)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=NUM_LABELS,
    )
    model.to(device)

    train_dataset = TextDataset(
        texts=train_df["text"].tolist(),
        labels=train_df["label"].tolist(),
        tokenizer=tokenizer,
        max_len=MAX_LEN,
    )
    val_dataset = TextDataset(
        texts=val_df["text"].tolist(),
        labels=val_df["label"].tolist(),
        tokenizer=tokenizer,
        max_len=MAX_LEN,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,   # на Mac — без multiprocessing
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    # считаем шаги оптимизатора (с учётом grad_accum)
    num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    total_optim_steps = num_update_steps_per_epoch * EPOCHS
    num_warmup_steps = int(WARMUP_RATIO * total_optim_steps)

    print(f"\nВсего optimizer steps: {total_optim_steps}, "
          f"warmup steps: {num_warmup_steps}")

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=total_optim_steps,
    )

    best_f1 = -1.0
    history = []
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for epoch in range(1, EPOCHS + 1):
        print(f"\n===== Эпоха {epoch}/{EPOCHS} =====")
        start_time = time.time()

        train_loss, train_acc, train_f1 = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            device,
            grad_accum_steps=GRAD_ACCUM_STEPS,
            epoch_idx=epoch,
        )
        val_loss, val_acc, val_f1 = eval_one_epoch(
            model,
            val_loader,
            device,
            epoch_idx=epoch,
        )

        elapsed = time.time() - start_time
        current_lr = scheduler.get_last_lr()[0]

        print(
            f"\nTrain | loss: {train_loss:.4f} | acc: {train_acc:.4f} | f1_macro: {train_f1:.4f}"
        )
        print(
            f"Val   | loss: {val_loss:.4f} | acc: {val_acc:.4f} | f1_macro: {val_f1:.4f}"
        )
        print(f"LR: {current_lr:.6f}")
        print(f"Время эпохи: {elapsed/60:.2f} мин")

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "train_f1": train_f1,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "val_f1": val_f1,
                "lr": current_lr,
                "time_min": elapsed / 60,
            }
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            print(f"🔥 Новый лучший F1: {best_f1:.4f}, сохраняем модель...")
            model.save_pretrained(OUTPUT_DIR)
            tokenizer.save_pretrained(OUTPUT_DIR)

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(
        os.path.join(OUTPUT_DIR, "training_history_rurorberta_base.csv"),
        index=False
    )

    print("\nОбучение завершено.")
    print(f"Лучшая macro-F1 на валидации: {best_f1:.4f}")
    print("Модель сохранена в:", OUTPUT_DIR)
    print("История обучения сохранена в:",
          os.path.join(OUTPUT_DIR, "training_history_rurorberta_base.csv"))


if __name__ == "__main__":
    main()


Читаем данные...
Размер train: (106311, 4)
Размер val:   (11771, 4)

Распределение классов (train):
label
0    35437
1    35437
2    35437
Name: count, dtype: int64

Распределение классов (val):
label
0    3896
1    3937
2    3938
Name: count, dtype: int64

Используем устройство: mps

Загружаем токенизатор и модель: DeepPavlov/rubert-base-cased


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Всего optimizer steps: 6646, warmup steps: 398

===== Эпоха 1/2 =====
[Train] Epoch 1 | batch 200/13289 (1.5%) | avg_loss: 1.1016 | lr: 0.000003 | elapsed: 1.1 мин
[Train] Epoch 1 | batch 400/13289 (3.0%) | avg_loss: 1.0926 | lr: 0.000005 | elapsed: 2.1 мин
[Train] Epoch 1 | batch 600/13289 (4.5%) | avg_loss: 1.0597 | lr: 0.000008 | elapsed: 3.1 мин
[Train] Epoch 1 | batch 800/13289 (6.0%) | avg_loss: 1.0205 | lr: 0.000010 | elapsed: 4.2 мин
[Train] Epoch 1 | batch 1000/13289 (7.5%) | avg_loss: 0.9879 | lr: 0.000013 | elapsed: 5.2 мин
[Train] Epoch 1 | batch 1200/13289 (9.0%) | avg_loss: 0.9582 | lr: 0.000015 | elapsed: 6.2 мин
[Train] Epoch 1 | batch 1400/13289 (10.5%) | avg_loss: 0.9334 | lr: 0.000018 | elapsed: 7.2 мин
[Train] Epoch 1 | batch 1600/13289 (12.0%) | avg_loss: 0.9123 | lr: 0.000020 | elapsed: 8.3 мин
[Train] Epoch 1 | batch 1800/13289 (13.5%) | avg_loss: 0.8994 | lr: 0.000020 | elapsed: 9.3 мин
[Train] Epoch 1 | batch 2000/13289 (15.1%) | avg_loss: 0.8867 | lr: 0.00002